In [1]:
"""
hr_attrition_xai_v2.py
══════════════════════════════════════════════════════════════════════════════
HR Attrition Prediction — Explainable & Ethical AI  (v2)
Hackathon: Trusted AI × HR
 
AMÉLIORATIONS vs v1
───────────────────
Data-cleaning
  • Détection automatique du leakage (AUC solo > 0.95)
  • Suppression colonnes *ID redondantes (GenderID == Sex, etc.)
  • Extraction de features métier AVANT anonymisation des dates
    → tenure_years, age_at_hire, current_age, hire_month, seniority_band
 
Feature engineering
  • attendance_risk     = DaysLateLast30 * 2 + Absences
  • disengagement_score = (5 - EmpSatisfaction) + (5 - EngagementSurvey)
  • salary_ratio        = Salary / médiane département
  • perf_risk_flag      = 1 si PerformanceScore ∈ {PIP, Needs Improvement}
 
Modèles
  • Logistic Regression, Random Forest, Gradient Boosting comparés
  • Sélection automatique du meilleur par CV-AUC (plus robuste que test-AUC)
  • OHE avec min_frequency=3 pour éviter explosion sur catégories rares
 
SHAP & rapports RH
  • Noms lisibles : "Score d'engagement" pas "EngagementSurvey"
  • Exclusion des colonnes OHE illisibles dans les rapports individuels
  • Valeurs formatées : $54,000 / 3.2/5 / 2 jours
 
Fairness
  • Ajout effet Cohen h (taille d'effet)
  • Légende couleur groupe référence vs autres
  • Marqueur étoile pour identifier le groupe référence
 
Run : python hr_attrition_xai_v2.py
"""
 
import re
import json
import hashlib
import warnings
from pathlib import Path
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import joblib
 
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    balanced_accuracy_score,
    classification_report,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
 
warnings.filterwarnings("ignore")
np.random.seed(42)
 
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
DATA_PATH = Path("data/HRDataset_v14.csv")
REF_DATE = pd.Timestamp("2024-01-01")
 
# ── optional deps ──────────────────────────────────────────────────────────
try:
    import shap
    SHAP_OK = True
    print("✅ SHAP disponible")
except ImportError:
    SHAP_OK = False
    print("⚠️  SHAP absent — pip install shap")
 
try:
    from codecarbon import EmissionsTracker
    CC_OK = True
    print("✅ CodeCarbon disponible")
except ImportError:
    CC_OK = False
 
# ══════════════════════════════════════════════════════════════════════════
# HELPERS
# ══════════════════════════════════════════════════════════════════════════
READABLE = {
    "Salary": "Salaire annuel ($)",
    "Absences": "Absences (jours)",
    "DaysLateLast30": "Retards sur 30 jours",
    "EmpSatisfaction": "Satisfaction (1-5)",
    "EngagementSurvey": "Engagement (1-5)",
    "SpecialProjectsCount": "Projets spéciaux",
    "tenure_years": "Ancienneté (années)",
    "age_at_hire": "Âge à l'embauche",
    "current_age": "Âge actuel",
    "hire_month": "Mois d'embauche",
    "attendance_risk": "Score assiduité (risque)",
    "disengagement_score": "Score désengagement",
    "salary_ratio": "Salaire vs médiane dép.",
    "perf_risk_flag": "Drapeau performance faible",
    "seniority_band": "Tranche d'ancienneté",
}
 
 
def to_readable(feat: str) -> str:
    if feat in READABLE:
        return READABLE[feat]
    m = re.match(r"^(.+?)_(.+)$", feat)
    if m:
        prefix = m.group(1).replace("_", " ").title()
        value = m.group(2).replace("_", " ")
        return f"{prefix} : {value}"
    return feat.replace("_", " ").title()
 
 
def fmt_value(col: str, val) -> str:
    if pd.isna(val):
        return "N/A"
    if "salary" in col.lower() or col == "Salary":
        return f"${int(val):,}"
    if col in ("EmpSatisfaction", "EngagementSurvey"):
        return f"{val:.1f}/5"
    if col in ("tenure_years", "age_at_hire", "current_age"):
        return f"{val:.1f} ans"
    if col in ("DaysLateLast30", "Absences"):
        return f"{int(val)} j"
    if col == "salary_ratio":
        return f"{val:.2f}×"
    if col == "perf_risk_flag":
        return "Oui" if val == 1 else "Non"
    return str(val)
 
 
def detect_leakage(df: pd.DataFrame, target: str, threshold: float = 0.95):
    y = df[target]
    leakers = []
    for col in df.columns:
        if col == target:
            continue
        try:
            x = df[col].copy()
            if x.dtype == "object":
                x = LabelEncoder().fit_transform(x.fillna("_NA_"))
            else:
                x = x.fillna(x.median())
            if x.std() == 0:
                continue
            auc = roc_auc_score(y, x)
            auc = max(auc, 1 - auc)
            if auc >= threshold:
                leakers.append((col, round(auc, 4)))
        except Exception:
            continue
    return leakers

✅ SHAP disponible
✅ CodeCarbon disponible


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("1. LOADING DATA")
print("="*60)
 
df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
print(f"Columns: {list(df_raw.columns)}")
print(f"Attrition rate: {df_raw['Termd'].mean():.1%}")


1. LOADING DATA
Loaded: 311 rows × 36 columns
Columns: ['Employee_Name', 'EmpID', 'MarriedID', 'MaritalStatusID', 'GenderID', 'EmpStatusID', 'DeptID', 'PerfScoreID', 'FromDiversityJobFairID', 'Salary', 'Termd', 'PositionID', 'Position', 'State', 'Zip', 'DOB', 'Sex', 'MaritalDesc', 'CitizenDesc', 'HispanicLatino', 'RaceDesc', 'DateofHire', 'DateofTermination', 'TermReason', 'EmploymentStatus', 'Department', 'ManagerName', 'ManagerID', 'RecruitmentSource', 'PerformanceScore', 'EngagementSurvey', 'EmpSatisfaction', 'SpecialProjectsCount', 'LastPerformanceReview_Date', 'DaysLateLast30', 'Absences']
Attrition rate: 33.4%


In [8]:
# ══════════════════════════════════════════════════════════════════════════
# 2. FEATURE ENGINEERING SUR DATES (avant anonymisation)
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 62)
print("2. FEATURE ENGINEERING SUR DATES")
print("═" * 62)
 
df = df_raw.copy()
 
if "DateofHire" in df.columns:
    hire = pd.to_datetime(df["DateofHire"], errors="coerce")
    df["tenure_years"] = (REF_DATE - hire).dt.days / 365.25
    df["hire_month"] = hire.dt.month
    df["seniority_band"] = pd.cut(
        df["tenure_years"].clip(lower=0),
        bins=[-0.1, 1, 3, 7, 100],
        labels=["<1 an", "1-3 ans", "3-7 ans", ">7 ans"],
    ).astype(str)
    print("  ✅ tenure_years, hire_month, seniority_band")
 
if "DOB" in df.columns and "DateofHire" in df.columns:
    dob = pd.to_datetime(df["DOB"], errors="coerce")
    hire = pd.to_datetime(df["DateofHire"], errors="coerce")
    df["age_at_hire"] = (hire - dob).dt.days / 365.25
    df["current_age"] = (REF_DATE - dob).dt.days / 365.25
    print("  ✅ age_at_hire, current_age")


══════════════════════════════════════════════════════════════
2. FEATURE ENGINEERING SUR DATES
══════════════════════════════════════════════════════════════
  ✅ tenure_years, hire_month, seniority_band
  ✅ age_at_hire, current_age


In [9]:
# ══════════════════════════════════════════════════════════════════════════
# 3. ANONYMISATION RGPD
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 62)
print("3. ANONYMISATION RGPD")
print("═" * 62)
 
DIRECT_IDS = [
    "Employee_Name", "EmpID", "ManagerName", "ManagerID",
    "DateofHire", "DateofTermination", "DOB",
    "LastPerformanceReview_Date",
]
dropped = [c for c in DIRECT_IDS if c in df.columns]
df.drop(columns=dropped, inplace=True)
 
if "Zip" in df.columns:
    df["Zip"] = df["Zip"].astype(str).str[:3] + "XX"
    print("  🔒 Codes postaux tronqués")
 
print(f"  🗑  Supprimés : {dropped}")
print(f"  ✅ {df.shape[1]} colonnes restantes")


══════════════════════════════════════════════════════════════
3. ANONYMISATION RGPD
══════════════════════════════════════════════════════════════
  🔒 Codes postaux tronqués
  🗑  Supprimés : ['Employee_Name', 'EmpID', 'ManagerName', 'ManagerID', 'DateofHire', 'DateofTermination', 'DOB', 'LastPerformanceReview_Date']
  ✅ 33 colonnes restantes


In [10]:
# ══════════════════════════════════════════════════════════════════════════
# 4. DÉTECTION LEAKAGE & SÉLECTION DES FEATURES
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 62)
print("4. DÉTECTION LEAKAGE & NETTOYAGE")
print("═" * 62)
 
TARGET = "Termd"
 
# Leakage dur — colonnes encodant directement l'outcome
HARD_LEAK = [
    "TermReason",        # raison de départ
    "EmploymentStatus",  # "Voluntarily Terminated", etc.
    "EmpStatusID",       # version numérique de EmploymentStatus
]
 
# IDs redondants — même information que leur version catégorielle
REDUNDANT_IDS = [
    "MarriedID",            # == MaritalDesc
    "MaritalStatusID",      # == MaritalDesc
    "GenderID",             # == Sex
    "DeptID",               # == Department
    "PerfScoreID",          # == PerformanceScore
    "PositionID",           # == Position
    "FromDiversityJobFairID",  # quasi-constant
]
 
# Sensibles — fairness uniquement, hors modèle
SENSITIVE = ["Sex", "RaceDesc", "HispanicLatino"]
 
# Détection automatique (AUC solo >= 0.95)
auto_leaks = detect_leakage(df, TARGET, threshold=0.95)
auto_leak_cols = [c for c, _ in auto_leaks]
 
if auto_leaks:
    print("\n  ⚠️  Leakage auto-détecté (AUC solo ≥ 0.95) :")
    for col, auc in auto_leaks:
        print(f"      {col:<35} AUC={auc}")
else:
    print("\n  ✅ Aucun leakage supplémentaire (seuil 0.95)")
 
ALL_EXCLUDE = set(HARD_LEAK + REDUNDANT_IDS + SENSITIVE + auto_leak_cols + [TARGET])
FEATURE_COLS = [c for c in df.columns if c not in ALL_EXCLUDE]
 
print(f"\n  Features retenues ({len(FEATURE_COLS)}) : {FEATURE_COLS}")
print(f"  Exclues leakage dur    : {[c for c in HARD_LEAK if c in df.columns]}")
print(f"  Exclues IDs redondants : {[c for c in REDUNDANT_IDS if c in df.columns]}")
print(f"  Sensibles (fairness)   : {[c for c in SENSITIVE if c in df.columns]}")
 
X = df[FEATURE_COLS].copy()
y = df[TARGET].copy()
sensitive_df = df[[c for c in SENSITIVE if c in df.columns]].copy()
 
# ── Feature engineering métier ──────────────────────────────────────────
print("\n  🔧 Feature engineering métier...")
 
if "DaysLateLast30" in X.columns and "Absences" in X.columns:
    X["attendance_risk"] = X["DaysLateLast30"] * 2 + X["Absences"]
    print("    attendance_risk = DaysLateLast30×2 + Absences")
 
if "EmpSatisfaction" in X.columns and "EngagementSurvey" in X.columns:
    X["disengagement_score"] = (
        (5 - X["EmpSatisfaction"].clip(1, 5))
        + (5 - X["EngagementSurvey"].clip(1, 5))
    )
    print("    disengagement_score = (5−Satisfaction) + (5−Engagement)")
 
if "Salary" in X.columns and "Department" in df.columns:
    dept_med = df.groupby("Department")["Salary"].transform("median").clip(lower=1)
    X["salary_ratio"] = X["Salary"] / dept_med
    print("    salary_ratio = Salary / médiane département")
 
if "PerformanceScore" in X.columns:
    low_perf = {"PIP", "Needs Improvement", "Needs Improvement "}
    X["perf_risk_flag"] = X["PerformanceScore"].isin(low_perf).astype(int)
    print("    perf_risk_flag = 1 si PerformanceScore ∈ {PIP, Needs Improvement}")
 
cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(include="number").columns.tolist()
print(f"\n  Numériques ({len(num_cols)}) : {num_cols}")
print(f"  Catégoriels ({len(cat_cols)}) : {cat_cols}")
 


══════════════════════════════════════════════════════════════
4. DÉTECTION LEAKAGE & NETTOYAGE
══════════════════════════════════════════════════════════════

  ⚠️  Leakage auto-détecté (AUC solo ≥ 0.95) :
      EmpStatusID                         AUC=0.9892

  Features retenues (19) : ['Salary', 'Position', 'State', 'Zip', 'MaritalDesc', 'CitizenDesc', 'Department', 'RecruitmentSource', 'PerformanceScore', 'EngagementSurvey', 'EmpSatisfaction', 'SpecialProjectsCount', 'DaysLateLast30', 'Absences', 'tenure_years', 'hire_month', 'seniority_band', 'age_at_hire', 'current_age']
  Exclues leakage dur    : ['TermReason', 'EmploymentStatus', 'EmpStatusID']
  Exclues IDs redondants : ['MarriedID', 'MaritalStatusID', 'GenderID', 'DeptID', 'PerfScoreID', 'PositionID', 'FromDiversityJobFairID']
  Sensibles (fairness)   : ['Sex', 'RaceDesc', 'HispanicLatino']

  🔧 Feature engineering métier...
    attendance_risk = DaysLateLast30×2 + Absences
    disengagement_score = (5−Satisfaction) + (5−Enga

In [11]:
# ══════════════════════════════════════════════════════════════════════════
# 5. SPLIT + PIPELINE DE PREPROCESSING
# ══════════════════════════════════════════════════════════════════════════
X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(
    X, y, sensitive_df, test_size=0.25, random_state=42, stratify=y
)
print(f"\n  Train : {X_train.shape}  |  Test : {X_test.shape}")
print(f"  Attrition — train : {y_train.mean():.1%}  |  test : {y_test.mean():.1%}")
 
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # min_frequency=3 : regroupe les catégories rares en "infrequent"
    ("ohe", OneHotEncoder(
        handle_unknown="infrequent_if_exist",
        sparse_output=False,
        min_frequency=3,
    )),
])
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
], remainder="drop")
 


  Train : (233, 23)  |  Test : (78, 23)
  Attrition — train : 33.5%  |  test : 33.3%


In [12]:
# ══════════════════════════════════════════════════════════════════════════
# 6. ENTRAÎNEMENT & COMPARAISON DES MODÈLES
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 62)
print("5. ENTRAÎNEMENT & COMPARAISON DES MODÈLES")
print("═" * 62)
 
if CC_OK:
    tracker = EmissionsTracker(
        project_name="hr_xai_v2",
        output_dir=str(OUTPUT_DIR),
        log_level="error",
    )
    tracker.start()
    print("  🌱 Carbon tracking démarré")
 
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
 
MODELS = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", C=0.3, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=4,
        min_samples_leaf=10,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=250,
        max_depth=3,
        learning_rate=0.04,
        subsample=0.8,
        min_samples_leaf=10,
        random_state=42,
    ),
}
 
results, pipes = {}, {}
 
for name, model in MODELS.items():
    pipe = Pipeline([("pre", preprocessor), ("clf", model)])
    pipe.fit(X_train, y_train)
 
    prob = pipe.predict_proba(X_test)[:, 1]
    pred = pipe.predict(X_test)
    auc = roc_auc_score(y_test, prob)
    bal = balanced_accuracy_score(y_test, pred)
    cv_s = cross_val_score(pipe, X, y, cv=cv5, scoring="roc_auc", n_jobs=-1)
 
    results[name] = dict(auc=auc, bal=bal, cv=cv_s, prob=prob, pred=pred)
    pipes[name] = pipe
    print(
        f"  {name:<25}  AUC={auc:.3f}  "
        f"BalAcc={bal:.3f}  CV={cv_s.mean():.3f}±{cv_s.std():.3f}"
    )
 
if CC_OK:
    em = tracker.stop()
    print(f"\n  🌿 Empreinte carbone : {em * 1e6:.4f} µgCO₂eq")
 
# Sélection du meilleur modèle par CV-AUC (plus robuste que test-AUC seul)
best_name = max(results, key=lambda k: results[k]["cv"].mean())
best = results[best_name]
best_pipe = pipes[best_name]
 
print(f"\n  ✅ Meilleur modèle : {best_name} (CV-AUC = {best['cv'].mean():.3f})")
print()
print(classification_report(y_test, best["pred"], target_names=["Actif", "Parti"]))
 
# Graphique comparaison modèles
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f"Comparaison des modèles  (meilleur : {best_name})",
    fontsize=13, fontweight="bold",
)
for name, r in results.items():
    RocCurveDisplay.from_predictions(
        y_test, r["prob"], ax=axes[0],
        name=f"{name}  AUC={r['auc']:.3f}",
    )
axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)
axes[0].set_title("Courbes ROC")
 
ConfusionMatrixDisplay.from_predictions(
    y_test, best["pred"],
    display_labels=["Actif", "Parti"],
    ax=axes[1], colorbar=False, cmap="Blues",
)
axes[1].set_title(f"Matrice de confusion — {best_name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_evaluation.png", dpi=150, bbox_inches="tight")
plt.close()
print("  ✅ Graphique comparaison sauvegardé")
 


══════════════════════════════════════════════════════════════
5. ENTRAÎNEMENT & COMPARAISON DES MODÈLES
══════════════════════════════════════════════════════════════
  🌱 Carbon tracking démarré
  Logistic Regression        AUC=0.760  BalAcc=0.721  CV=0.747±0.078
  Random Forest              AUC=0.814  BalAcc=0.750  CV=0.779±0.075
  Gradient Boosting          AUC=0.776  BalAcc=0.702  CV=0.792±0.046

  🌿 Empreinte carbone : 4.2797 µgCO₂eq

  ✅ Meilleur modèle : Gradient Boosting (CV-AUC = 0.792)

              precision    recall  f1-score   support

       Actif       0.81      0.75      0.78        52
       Parti       0.57      0.65      0.61        26

    accuracy                           0.72        78
   macro avg       0.69      0.70      0.69        78
weighted avg       0.73      0.72      0.72        78

  ✅ Graphique comparaison sauvegardé


In [ ]:
# ══════════════════════════════════════════════════════════════════════════
# 7. EXPLICABILITÉ — SHAP
# ══════════════════════════════════════════════════════════════════════════
print("\n" + "═" * 62)
print("6. EXPLICABILITÉ (SHAP)")
print("═" * 62)
 
ohe_obj = best_pipe.named_steps["pre"].named_transformers_["cat"]["ohe"]
ohe_features = ohe_obj.get_feature_names_out(cat_cols).tolist()
all_feats = num_cols + ohe_features
all_readable = [to_readable(f) for f in all_feats]
 
X_test_proc = best_pipe.named_steps["pre"].transform(X_test)
best_model = best_pipe.named_steps["clf"]
 
if SHAP_OK:
    print("  Calcul des valeurs SHAP...")
    if hasattr(best_model, "estimators_"):
        explainer = shap.TreeExplainer(best_model)
        sv_raw = explainer.shap_values(X_test_proc)
        
        # 1. Handle SHAP values
        if isinstance(sv_raw, list):
            # Traditional Random Forest / LightGBM often returns a list [class_0, class_1]
            sv = sv_raw[1]
        elif len(sv_raw.shape) == 3:
            # Some versions return (samples, features, classes)
            sv = sv_raw[:, :, 1]
        else:
            # It's already 2D (samples, features) for the positive class or regression
            sv = sv_raw

        # 2. Handle Expected Value (Base Value)
        ev = explainer.expected_value
        if hasattr(ev, "__len__") and len(ev) > 1:
            base_val = ev[1]
        else:
            # It's a single scalar
            base_val = ev
    else:
        explainer = shap.LinearExplainer(best_model, X_test_proc)
        sv = explainer.shap_values(X_test_proc)
        if isinstance(sv, list):
            sv = sv[1]
        base_val = explainer.expected_value
 
    fi_df = (
        pd.DataFrame({
            "feature": all_feats,
            "readable": all_readable,
            "importance": np.abs(sv).mean(axis=0),
        })
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
 
    # Bar global
    top15 = fi_df.head(15).sort_values("importance")
    colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(top15)))
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(top15["readable"], top15["importance"], color=colors, alpha=0.88)
    ax.set_xlabel("Importance SHAP moyenne  |SHAP|", fontsize=12)
    ax.set_title("Top 15 facteurs de risque d'attrition", fontsize=13, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "shap_global_bar.png", dpi=150, bbox_inches="tight")
    plt.close()
 
    # Beeswarm
    exp_all = shap.Explanation(
        values=sv,
        base_values=np.full(len(sv), base_val),
        data=X_test_proc,
        feature_names=all_readable,
    )
    plt.figure(figsize=(11, 7))
    shap.plots.beeswarm(exp_all, max_display=15, show=False)
    plt.title("Distribution de l'impact SHAP par feature", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "shap_beeswarm.png", dpi=150, bbox_inches="tight")
    plt.close()
 
    # Waterfall employé à risque maximal
    top_idx = best["prob"].argmax()
    exp_loc = shap.Explanation(
        values=sv[top_idx],
        base_values=base_val,
        data=X_test_proc[top_idx],
        feature_names=all_readable,
    )
        # On extrait la valeur de base pour la classe 1 (le risque)
    # Si c'est un tableau, on prend l'index [1], sinon on garde la valeur telle quelle
    # APRÈS (corrigé)
    bv = base_val[1] if hasattr(base_val, "__len__") and len(base_val) > 1 else base_val
    bv_scalar = float(np.array(bv).ravel()[0])


    exp_loc = shap.Explanation(
        values=sv[top_idx],              # Vecteur 1D des contributions
        base_values=bv_scalar,           # LE SCALAIRE (ex: 0.24)
        # APRÈS
        data=X_test_proc[top_idx],
        feature_names=all_readable,  # Valeurs réelles des variables
)

    # Ton code de plotting tel quel :
    plt.figure(figsize=(11, 6))
    shap.plots.waterfall(exp_loc, show=False)
    plt.title(
        f"Explication locale — employé à risque max (p={best['prob'][top_idx]:.1%})",
        fontsize=12, fontweight="bold",
    )
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "shap_local_waterfall.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  ✅ Graphique SHAP sauvegardé")
 
else:
    print("  Importance par permutation (fallback SHAP)...")
    perm = permutation_importance(
        best_model, X_test_proc, y_test, n_repeats=30, random_state=42, n_jobs=-1
    )
    fi_df = (
        pd.DataFrame({
            "feature": all_feats,
            "readable": all_readable,
            "importance": perm.importances_mean,
        })
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    top15 = fi_df.head(15).sort_values("importance")
    colors = plt.cm.RdYlGn_r(np.linspace(0.1, 0.9, len(top15)))
    fig, ax = plt.subplots(figsize=(11, 7))
    ax.barh(top15["readable"], top15["importance"], color=colors, alpha=0.88)
    ax.set_xlabel("Importance par permutation", fontsize=12)
    ax.set_title("Top 15 facteurs de risque d'attrition", fontsize=13, fontweight="bold")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "feature_importance.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  ✅ Graphique importance sauvegardé")
 
print("\n  Top 5 facteurs d'attrition :")
for _, row in fi_df.head(5).iterrows():
    print(f"    {row['readable']:<40}  {row['importance']:.4f}")
 


══════════════════════════════════════════════════════════════
6. EXPLICABILITÉ (SHAP)
══════════════════════════════════════════════════════════════
  Calcul des valeurs SHAP...


TypeError: only 0-dimensional arrays can be converted to Python scalars

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# 8. FAIRNESS AUDIT
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("6. FAIRNESS AUDIT")
print("="*60)
 
def fairness_metrics(y_true, y_pred, groups, privileged):
    results = []
    y_true, y_pred, groups = np.array(y_true), np.array(y_pred), np.array(groups)
    priv = groups == privileged
    priv_pos = y_pred[priv].mean() if priv.sum() > 0 else np.nan
    priv_tpr = y_pred[priv & (y_true == 1)].mean() if (priv & (y_true == 1)).sum() > 0 else np.nan
 
    for g in np.unique(groups):
        mask = groups == g
        pos = y_pred[mask].mean()
        tpr = y_pred[mask & (y_true == 1)].mean() if (mask & (y_true == 1)).sum() > 0 else np.nan
        di = pos / priv_pos if priv_pos and priv_pos > 0 else np.nan
        spd = pos - priv_pos
        eod = tpr - priv_tpr if not (np.isnan(tpr) or np.isnan(priv_tpr)) else np.nan
        results.append({
            'group': g, 'privileged': g == privileged,
            'n': mask.sum(), 'pos_rate': pos, 'tpr': tpr,
            'disparate_impact': di,
            'stat_parity_diff': spd,
            'equal_opp_diff': eod,
            'di_fair': 0.8 <= di <= 1.25 if not np.isnan(di) else False,
            'spd_fair': abs(spd) <= 0.1,
            'eod_fair': abs(eod) <= 0.1 if not np.isnan(eod) else False,
        })
    return pd.DataFrame(results)
 
fairness_results = {}
y_test_arr, rf_pred_arr = y_test.values, rf_pred
 
for attr, col, privileged in [('Gender', 'Sex', 'M'), ('Race', 'RaceDesc', 'White')]:
    if col not in s_test.columns:
        continue
    series = s_test[col].values
    if privileged not in series:
        privileged = pd.Series(series).value_counts().index[0]
    fdf = fairness_metrics(y_test_arr, rf_pred_arr, series, privileged)
    fairness_results[attr] = fdf
 
    print(f"\n  📊 {attr} fairness (privileged: {privileged})")
    print(f"  {'Group':<30} {'n':>5} {'PosRate':>8} {'DI':>7} {'SPD':>7} {'EOD':>7}")
    print("  " + "-"*65)
    for _, r in fdf.iterrows():
        di_s = f"{r['disparate_impact']:.3f}" if not np.isnan(r['disparate_impact']) else " N/A"
        spd_s = f"{r['stat_parity_diff']:+.3f}"
        eod_s = f"{r['equal_opp_diff']:+.3f}" if not np.isnan(r['equal_opp_diff']) else "  N/A"
        di_flag = "✅" if r['di_fair'] else "⚠️ "
        spd_flag = "✅" if r['spd_fair'] else "⚠️ "
        print(f"  {r['group']:<30} {r['n']:>5} {r['pos_rate']:>8.3f} {di_s:>6}{di_flag} {spd_s:>6}{spd_flag} {eod_s:>7}")
 
# Fairness chart
if fairness_results:
    fig, axes = plt.subplots(1, len(fairness_results), figsize=(7 * len(fairness_results), 5))
    if len(fairness_results) == 1:
        axes = [axes]
    for ax, (attr, fdf) in zip(axes, fairness_results.items()):
        x = np.arange(len(fdf))
        w = 0.35
        actual_rates = [fdf[fdf['group']==g]['pos_rate'].values[0] if len(fdf[fdf['group']==g]) > 0 else 0
                       for g in fdf['group']]
        ax.bar(x - w/2, fdf['pos_rate'], w, label='Predicted rate', color='#e74c3c', alpha=0.7)
        ax.set_xticks(x)
        ax.set_xticklabels(fdf['group'], rotation=30, ha='right', fontsize=9)
        ax.set_title(f'{attr} — Prediction Rate by Group', fontsize=12)
        ax.set_ylabel('Attrition Prediction Rate')
        ax.set_ylim(0, 1)
        max_rate = fdf['pos_rate'].max()
        ax.axhline(0.8 * max_rate, color='orange', linestyle='--',
                   alpha=0.7, label='80% rule boundary')
        ax.legend(fontsize=9)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
 
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'fairness_audit.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("\n  ✅ Fairness plot saved")


6. FAIRNESS AUDIT

  📊 Gender fairness (privileged: F)
  Group                              n  PosRate      DI     SPD     EOD
  -----------------------------------------------------------------
  F                                 34    0.294  1.000✅ +0.000✅  +0.000
  M                                 29    0.345  1.172✅ +0.051✅  +0.091

  📊 Race fairness (privileged: White)
  Group                              n  PosRate      DI     SPD     EOD
  -----------------------------------------------------------------
  American Indian or Alaska Native     1    0.000  0.000⚠️  -0.344⚠️      N/A
  Asian                              7    0.429  1.247✅ +0.085✅  +0.000
  Black or African American         20    0.250  0.727⚠️  -0.094✅  -0.167
  Two or more races                  3    0.333  0.970✅ -0.010✅  +0.000
  White                             32    0.344  1.000✅ +0.000✅  +0.000

  ✅ Fairness plot saved


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# 9. HR RISK REPORTS
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("7. HR RISK REPORTS")
print("="*60)
 
top3 = rf_prob.argsort()[-3:][::-1]
reports = []
for idx in top3:
    p = rf_prob[idx]
    risk = 'HIGH' if p > 0.6 else ('MEDIUM' if p > 0.35 else 'LOW')
    emoji = {'HIGH': '🔴', 'MEDIUM': '🟡', 'LOW': '🟢'}[risk]
    top_fi = fi_df.head(5)
    emp = X_test.iloc[idx]
 
    recs = {
        'HIGH': ['Schedule 1:1 retention conversation ASAP',
                 'Review compensation vs. market benchmark',
                 'Assign mentor / executive sponsor',
                 'Explore internal mobility opportunities'],
        'MEDIUM': ['Schedule check-in this month',
                   'Review engagement survey responses',
                   'Discuss career development path'],
        'LOW': ['Continue regular check-ins',
                'Monitor engagement trends quarterly']
    }
 
    lines = [
        "=" * 65,
        f"  ATTRITION RISK REPORT — CONFIDENTIAL  (GDPR protected)",
        "=" * 65,
        f"  Employee:  #{''.join([c for c in hashlib.sha256(str(idx).encode()).hexdigest()[:8]])} (pseudonymized)",
        f"  Risk:      {emoji} {risk}",
        f"  Prob:      {p:.1%}",
        "",
        "  KEY RISK FACTORS:",
    ]
    for _, row in top_fi.iterrows():
        val = emp.get(row['feature'], 'N/A')
        lines.append(f"    • {row['feature']:<28} value: {val}  (weight: {row['importance']:.3f})")
 
    lines += ["", "  RECOMMENDED ACTIONS:"]
    for rec in recs[risk]:
        lines.append(f"    → {rec}")
    lines += [
        "",
        "  ⚠️  This is a decision-support tool, not a final verdict.",
        "      Always apply human judgment before taking any action.",
        "=" * 65
    ]
    report = "\n".join(lines)
    reports.append(report)
    print(report)
 
with open(OUTPUT_DIR / 'hr_risk_reports.txt', 'w', encoding='utf-8') as f:
    f.write("\n\n".join(reports))

print("\n  ✅ HR reports saved")


7. HR RISK REPORTS
  ATTRITION RISK REPORT — CONFIDENTIAL  (GDPR protected)
  Employee:  #1a656259 (pseudonymized)
  Risk:      🔴 HIGH
  Prob:      74.7%

  KEY RISK FACTORS:
    • EmpStatusID                  value: 5  (weight: 0.128)
    • RecruitmentSource_Google Search value: N/A  (weight: 0.016)
    • MaritalDesc_Single           value: N/A  (weight: 0.012)
    • RecruitmentSource_LinkedIn   value: N/A  (weight: 0.011)
    • Salary                       value: 54005  (weight: 0.009)

  RECOMMENDED ACTIONS:
    → Schedule 1:1 retention conversation ASAP
    → Review compensation vs. market benchmark
    → Assign mentor / executive sponsor
    → Explore internal mobility opportunities

  ⚠️  This is a decision-support tool, not a final verdict.
      Always apply human judgment before taking any action.
  ATTRITION RISK REPORT — CONFIDENTIAL  (GDPR protected)
  Employee:  #19581e27 (pseudonymized)
  Risk:      🔴 HIGH
  Prob:      74.2%

  KEY RISK FACTORS:
    • EmpStatusID        

In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# 10. SAVE MODEL & METRICS
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("8. SAVING ARTIFACTS")
print("="*60)
 
joblib.dump(rf_pipe, OUTPUT_DIR / 'rf_attrition_model.joblib')
print("  ✅ Model saved: rf_attrition_model.joblib")
 
metrics = {
    'model': 'RandomForest',
    'hyperparams': {'n_estimators': 200, 'max_depth': 6, 'min_samples_leaf': 5},
    'auc_roc': float(rf_auc),
    'lr_baseline_auc': float(lr_auc),
    'cv_auc_mean': float(cv_scores.mean()),
    'cv_auc_std': float(cv_scores.std()),
    'test_set_size': int(len(y_test)),
    'attrition_rate_test': float(y_test.mean()),
    'leakage_cols_excluded': LEAKAGE_COLS,
    'sensitive_cols_excluded_from_model': SENSITIVE_COLS,
    'features_used': FEATURE_COLS,
}
with open(OUTPUT_DIR / 'metrics_summary.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("  ✅ Metrics saved: metrics_summary.json")
 
print("\n" + "="*60)
print("✅  ALL DONE")
print("="*60)
print("\nOutputs generated:")
for fp in sorted(OUTPUT_DIR.iterdir()):
    size = fp.stat().st_size
    print(f"  📄 {fp.name:<40} ({size/1024:.1f} KB)")


8. SAVING ARTIFACTS
  ✅ Model saved: rf_attrition_model.joblib
  ✅ Metrics saved: metrics_summary.json

✅  ALL DONE

Outputs generated:
  📄 emissions.csv                            (1.0 KB)
  📄 fairness_audit.png                       (77.8 KB)
  📄 hr_risk_reports.txt                      (3.1 KB)
  📄 metrics_summary.json                     (1.0 KB)
  📄 model_evaluation.png                     (81.5 KB)
  📄 rf_attrition_model.joblib                (375.7 KB)
  📄 shap_beeswarm.png                        (132.7 KB)
  📄 shap_global_bar.png                      (110.6 KB)
  📄 shap_local_waterfall.png                 (107.9 KB)
